#### Download the dataset from Kaggle, add to current repo

In [50]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('titanic')

print("Path to competition files:", path)

Path to competition files: /Users/swanynguyen/.cache/kagglehub/competitions/titanic


#### Load Data

In [51]:
import pandas as pd

# Load data
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
print(train.head(10))

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   
5            6         0       3   
6            7         0       1   
7            8         0       3   
8            9         1       3   
9           10         1       2   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   
5                                   Moran, Mr. James    male   NaN      0   
6                            McCarthy, Mr. Timothy J    male  54

#### Observe dataset and check for Null and duplicated values


In [52]:
# print(train.describe())
print(train.info())
print(f"Columns with missing values:, {train.columns[train.isnull().any()]}")
# Check for duplicates
print(f"Total duplicated rows: {train.duplicated().sum()}")

#

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None
Columns with missing values:, Index(['Age', 'Cabin', 'Embarked'], dtype='object')
Total duplicated rows: 0


#### Handle missing values

In [ ]:
# Handle missing values
# Fill 'Embarked' with mode(), most common port of embarkation
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])
# Fill 'Age' with median(), the mid range age of passenger
train['Age'] = train['Age'].fillna(train['Age'].median())
"""The `Cabin` values have a first letter followed by a number. 
The letters indicate the deck level, while the numbers indicate the room numbers. 
We create a new column storing the `Deck` information, which contains only the first letter. 
We then drop the `Cabin` column since it contains many missing values, and filling them would not necessarily benefit the accuracy of the model."""
train['Deck']= train['Cabin'].str[0].fillna('Unkown')
print(f"Missing Values:, {train.isnull().sum()}")


Missing Values:, PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
Deck             0
dtype: int64


In [54]:
# Drop irrelevant columns
train = train.drop(columns=['Name','Cabin','Ticket', 'PassengerId'])

In [55]:
# Encode categorical variables to numerical for modelling
train = pd.get_dummies(train, columns=['Sex', 'Embarked', 'Deck'],dtype=int)
train.info()
train.head(2)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 20 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Survived     891 non-null    int64  
 1   Pclass       891 non-null    int64  
 2   Age          891 non-null    float64
 3   SibSp        891 non-null    int64  
 4   Parch        891 non-null    int64  
 5   Fare         891 non-null    float64
 6   Sex_female   891 non-null    int64  
 7   Sex_male     891 non-null    int64  
 8   Embarked_C   891 non-null    int64  
 9   Embarked_Q   891 non-null    int64  
 10  Embarked_S   891 non-null    int64  
 11  Deck_A       891 non-null    int64  
 12  Deck_B       891 non-null    int64  
 13  Deck_C       891 non-null    int64  
 14  Deck_D       891 non-null    int64  
 15  Deck_E       891 non-null    int64  
 16  Deck_F       891 non-null    int64  
 17  Deck_G       891 non-null    int64  
 18  Deck_T       891 non-null    int64  
 19  Deck_Unk

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T,Deck_Unkown
0,0,3,22.0,1,0,7.2500,0,1,0,0,1,0,0,0,0,0,0,0,0,1
1,1,1,38.0,1,0,71.2833,1,0,1,0,0,0,0,1,0,0,0,0,0,0


#### Explorative Analysis

In [63]:
# Fare of survival rate
print(f"Maximum Fare: ${train['Fare'].max()}")
print(f"Minimum Fare: ${train['Fare'].min()}")

# 1. Define custom fare intervals (or let pandas choose using an integer)
# Bins: 0-20, 20-100, 100-200, 200-600
fare_bins = [0, 20, 100, 200, 600]
fare_labels = ['Budget (<20)', 'Economy (20-100)', 'Business (100-200)', 'Luxury (200+)']

# 2. Create the fare range column
train['Fare_Range'] = pd.cut(train['Fare'], bins=fare_bins, labels=fare_labels, include_lowest=True)

budget = train.loc[train.Fare_Range == 'Budget (<20)','Survived']
budget_survive = budget.mean()
econ = train.loc[train.Fare_Range == 'Economy (20-100)','Survived']
econ_survive = econ.mean()
business = train.loc[train.Fare_Range == 'Business (100-200)','Survived']
business_survive = business.mean()
luxury = train.loc[train.Fare_Range == 'Luxury (200+)','Survived']
luxury_survive = luxury.mean()

print("% of People who got 'Budget' fare that survived:", budget_survive)
print("% of People who got 'Economy' fare that survived:", econ_survive)
print("% of People who got 'Business' fare that survived:", business_survive)
print("% of People who got 'Luxury' fare that survived:", luxury_survive)


Maximum Fare: $512.3292
Minimum Fare: $0.0
% of People who got 'Budget' fare that survived: 0.27766990291262134
% of People who got 'Economy' fare that survived: 0.4953560371517028
% of People who got 'Business' fare that survived: 0.7575757575757576
% of People who got 'Luxury' fare that survived: 0.7
